# Bank Customer Churn - Exploratory Data Analysis

This notebook documents EDA for the churn dataset and links the results to business interpretation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

project_root = Path.cwd().parent
raw_path = project_root / "data" / "raw" / "Bank Customer Churn Prediction.csv"
processed_path = project_root / "data" / "processed" / "processed_bank_churn.csv"
figures_dir = project_root / "reports" / "figures" / "intermediate"
metrics_dir = project_root / "reports" / "metrics"

df = pd.read_csv(processed_path if processed_path.exists() else raw_path)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

figures_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

df.head()

## Dataset Health Checks
This section covers shape, missing values, duplicate rows, target distribution, and churn rate.

In [ ]:
shape_df = pd.DataFrame([{"rows": df.shape[0], "columns": df.shape[1]}])
missing_df = df.isna().sum().reset_index()
missing_df.columns = ["column", "missing_count"]
duplicates = int(df.duplicated().sum())
target_counts = df["churn"].value_counts(dropna=False).rename_axis("churn").reset_index(name="count")
churn_rate = float(df["churn"].mean())

overview_df = pd.DataFrame([{
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "duplicate_rows": duplicates,
    "churn_rate": churn_rate
}])

overview_df.to_csv(metrics_dir / "eda_overview.csv", index=False)
missing_df.to_csv(metrics_dir / "eda_missing_values.csv", index=False)

print("Dataset shape:", tuple(df.shape))
print("Duplicate rows:", duplicates)
print("Churn rate:", round(churn_rate, 4))

display(overview_df)
display(target_counts)
display(missing_df.sort_values("missing_count", ascending=False).head(12))

## Numeric and Categorical Distributions

In [ ]:
numeric_cols = [
    "credit_score", "age", "tenure", "balance", "products_number",
    "estimated_salary", "balance_to_salary_ratio", "products_per_tenure"
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

cat_cols = ["country", "gender", "credit_card", "active_member", "products_number"]
cat_cols = [c for c in cat_cols if c in df.columns]

numeric_summary = df[numeric_cols].describe().T
numeric_summary.to_csv(metrics_dir / "eda_numeric_summary.csv")
display(numeric_summary)

cat_parts = []
for col in cat_cols:
    grouped = df.groupby(col, as_index=False).agg(customer_count=("churn", "size"), churn_rate=("churn", "mean"))
    grouped.insert(0, "feature", col)
    grouped = grouped.rename(columns={col: "category_value"})
    cat_parts.append(grouped)

categorical_summary = pd.concat(cat_parts, ignore_index=True)
categorical_summary.to_csv(metrics_dir / "eda_categorical_summary.csv", index=False)
display(categorical_summary.head(20))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
plot_cols = numeric_cols[:6]
for idx, col in enumerate(plot_cols):
    sns.histplot(df[col], kde=True, ax=axes[idx], color="steelblue")
    axes[idx].set_title(f"Distribution - {col}")

for idx in range(len(plot_cols), len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

## Churn-Focused Plots Required by Project Scope

In [ ]:
fig_paths = [
    "target_distribution.png",
    "churn_by_country.png",
    "churn_by_gender.png",
    "churn_by_active_member.png",
    "churn_by_products_number.png",
    "age_vs_churn.png",
    "balance_vs_churn.png",
    "correlation_heatmap.png",
]

n = len(fig_paths)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, name in enumerate(fig_paths):
    img_path = figures_dir / name
    axes[i].set_title(name.replace("_", " ").replace(".png", "").title())
    if img_path.exists():
        img = plt.imread(img_path)
        axes[i].imshow(img)
        axes[i].axis("off")
    else:
        axes[i].text(0.5, 0.5, f"Missing: {name}", ha="center", va="center")
        axes[i].axis("off")

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## Business Interpretation

- Customer segments with higher churn rates are priority targets for retention interventions.
- Churn differences by country, active status, and number of products can guide segmented campaign design.
- Older customers and customers with higher balances should be reviewed jointly with engagement behavior to avoid over-generalization.
- Correlation patterns are directional signals only; they do not imply causation.
- This analysis supports decision making for retention strategy, and should be paired with model validation and cost-impact analysis.